<a href="https://colab.research.google.com/github/alicsrsustain-sudo/HVAC-Optimization-/blob/main/Pump_Speed_Reset_Strategy_(Saving).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# INPUTS
# ==========================================

FILE_PATH = r"pump_speed_data.csv"
+


PUMP_RATED_POWER_KW = 5.5
ELECTRICITY_TARIFF = 0.2225

SPEED_COLUMN = "Value"

INTERVAL_HOURS = 0.25  # 15 min data

# ==========================================
# YOUR PROPOSED STRATEGY
# ==========================================

SPEED_LEVELS = {
    "low": 50,
    "med_low": 70,
    "med_high": 80,
    "peak": 90
}

# ==========================================
# LOAD DATA
# ==========================================

df = pd.read_csv(FILE_PATH)

df = df.dropna(subset=[SPEED_COLUMN])

df[SPEED_COLUMN] = (
    df[SPEED_COLUMN]
    .astype(str)
    .str.replace("%", "", regex=False)
)

df[SPEED_COLUMN] = pd.to_numeric(df[SPEED_COLUMN], errors="coerce")
df = df.dropna(subset=[SPEED_COLUMN])

# ==========================================
# BASELINE (REAL OPERATION)
# ==========================================

df["Baseline_Power_kW"] = (
    PUMP_RATED_POWER_KW *
    (df[SPEED_COLUMN] / 100) ** 3
)

df["Baseline_Energy_kWh"] = df["Baseline_Power_kW"] * INTERVAL_HOURS

baseline_energy = df["Baseline_Energy_kWh"].sum()
baseline_cost = baseline_energy * ELECTRICITY_TARIFF

# ==========================================
# PROPOSED STRATEGY
# (map real demand → 4 bands)
# ==========================================

# We use percentiles of actual operation to define demand bands
q1 = np.percentile(df[SPEED_COLUMN], 25)
q2 = np.percentile(df[SPEED_COLUMN], 50)
q3 = np.percentile(df[SPEED_COLUMN], 75)

def map_speed(speed):
    if speed <= q1:
        return SPEED_LEVELS["low"]
    elif speed <= q2:
        return SPEED_LEVELS["med_low"]
    elif speed <= q3:
        return SPEED_LEVELS["med_high"]
    else:
        return SPEED_LEVELS["peak"]

df["Scenario_Speed"] = df[SPEED_COLUMN].apply(map_speed)

df["Scenario_Power_kW"] = (
    PUMP_RATED_POWER_KW *
    (df["Scenario_Speed"] / 100) ** 3
)

df["Scenario_Energy_kWh"] = df["Scenario_Power_kW"] * INTERVAL_HOURS

scenario_energy = df["Scenario_Energy_kWh"].sum()
scenario_cost = scenario_energy * ELECTRICITY_TARIFF

# ==========================================
# SAVINGS
# ==========================================

saving = baseline_cost - scenario_cost
saving_pct = (saving / baseline_cost) * 100

# ==========================================
# RESULTS
# ==========================================

print("\nRESULTS")
print("=" * 60)

print(f"Baseline Cost: £{baseline_cost:,.2f}")
print(f"4-Step Strategy Cost: £{scenario_cost:,.2f}")

print(f"\nAnnual Saving: £{saving:,.2f}")
print(f"Saving Percentage: {saving_pct:.1f}%")

print("\nProposed Speed Levels Used:")
for k, v in SPEED_LEVELS.items():
    print(f"{k}: {v}%")

FileNotFoundError: [Errno 2] No such file or directory: 'pump_speed_data.csv'